In [1]:
# Import libraries for text processing,
# dimensionality reduction, classification, and validation

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Load the preprocessed and reduced dataset
# This is a subset of the Sentiment140 corpus with texts and labels

df_reduced = pd.read_csv('twitter_reduced_Dataset.csv', encoding='latin-1')

# Split the predictor variables (texts) and the target variable (sentiment)
X = df_reduced['text'].values
y = df_reduced['target'].values

# Define a custom stopword list
# These words will be filtered out during vectorization because they do not provide relevant information

custom_stopwords = [
    'as', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
    'is', 'it', 'for', 'with', 'that', 'this', 'was', 'be',
    'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
    'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not'
]


In [2]:
i=0
# Definition of parameters to explore:
# k: number of topics to extract using LDA
# C: regularization parameter for logistic regression

k_values = [10]
C_values = [1]

# Initialize the list where the results will be stored
results = []

# Configure 5-fold cross-validation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Main experimentation loop for each combination (k, C)

for k in k_values:
    for C in C_values:
        scores = []  # List to store the scores of each fold

        # Cross-validation loop
        for train_idx, test_idx in cv.split(X, y):
            # Extract texts and labels for training and testing
            X_train_raw, X_test_raw = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Text vectorization with CountVectorizer
            # It is fitted only on the training set to avoid data leakage

            vectorizer = CountVectorizer(max_features=10000, stop_words=custom_stopwords)
            X_train_counts = vectorizer.fit_transform(X_train_raw)
            X_test_counts = vectorizer.transform(X_test_raw)

            # Dimensionality reduction with LDA (Latent Dirichlet Allocation)
            # k latent topics are extracted from the training set

            lda = LatentDirichletAllocation(n_components=k, random_state=42)
            X_train_topics = lda.fit_transform(X_train_counts)
            X_test_topics = lda.transform(X_test_counts)

            # Train the logistic regression model
            # L2 regularization is used

            model = LogisticRegression(
                penalty='l2',
                C=C,
                solver='lbfgs',
                max_iter=100,
                random_state=42,
            )
            model.fit(X_train_topics, y_train)
            print("Iteration: "+str(i) )
            i=i+1

            # Predict probabilities and compute the AUC-ROC metric
            # This metric evaluates the model's discriminative ability

            probabilities = model.predict_proba(X_test_topics)[:, 1]
            score = roc_auc_score(y_test, probabilities)
            scores.append(score)
            print("AUC-ROC: Fold " + str(i) +  ":" + str(score))  # Print the AUC score for the current fold

        # Store the mean result for the current parameter combination

        results.append({
            'k': k,
            'C': C,
            'mean_auc_roc': np.mean(scores)
        })

        # Print partial results to monitor progress
        print(results)


Iteration: 0
AUC-ROC: Fold 1:0.58601328125
Iteration: 1
AUC-ROC: Fold 2:0.6121251953125
Iteration: 2
AUC-ROC: Fold 3:0.6098244140625
Iteration: 3
AUC-ROC: Fold 4:0.5948289062500001
Iteration: 4
AUC-ROC: Fold 5:0.61718828125
[{'k': 10, 'C': 1, 'mean_auc_roc': np.float64(0.603996015625)}]


In [3]:
# Organize the results
df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=10):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for k=10):

 k  C  mean_auc_roc
10  1        0.6040
